# 🔲 Convolution Operation — Notes + Interview
---
> **Simple English** | **Interview Ready** | Core of all CNN models

## 📌 What is Convolution? (Simple English)
- Convolution = sliding a small **filter (kernel)** over an image to detect features
- The filter looks for patterns like **edges, curves, textures**
- At each position: multiply filter values × image values → sum them up = 1 number
- This creates a new smaller grid called a **Feature Map**
- Key idea: the same filter scans the **entire image** → shares weights

## 🔑 Why Not Just Use Fully Connected (Dense) Layers for Images?
| Problem | Dense Layer | Convolution |
|---|---|---|
| 224×224 image params | 150,000+ per neuron ❌ | Tiny filter (e.g. 3×3=9) ✅ |
| Learns position | Yes (can't generalize) | No (position-independent) ✅ |
| Translation invariance | ❌ | ✅ |

## 🧱 Convolution Formula
```
Feature_Map[i,j] = Σ Σ Image[i+m, j+n] × Filter[m,n]
```
- Slide filter over every position → compute dot product → build feature map

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Manual 2D Convolution ──
def conv2d(image, kernel):
    ih, iw = image.shape
    kh, kw = kernel.shape
    oh, ow = ih - kh + 1, iw - kw + 1
    output = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            output[i, j] = np.sum(image[i:i+kh, j:j+kw] * kernel)
    return output

# 5×5 image
image = np.array([
    [1, 2, 3, 0, 1],
    [0, 1, 2, 3, 1],
    [1, 0, 1, 2, 0],
    [2, 1, 0, 1, 2],
    [0, 2, 1, 0, 1]
], dtype=float)

# Edge detection kernel (Sobel-like)
kernel_edge = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1]
], dtype=float)

# Blur kernel
kernel_blur = np.ones((3,3)) / 9

feature_edge = conv2d(image, kernel_edge)
feature_blur = conv2d(image, kernel_blur)

fig, axes = plt.subplots(1, 3, figsize=(12,4))
axes[0].imshow(image, cmap='gray'); axes[0].set_title('Original Image (5×5)')
axes[1].imshow(feature_edge, cmap='gray'); axes[1].set_title('Edge Detection Filter')
axes[2].imshow(feature_blur, cmap='gray'); axes[2].set_title('Blur Filter')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()
print(f"Input: {image.shape} → Output: {feature_edge.shape}  (5×5 with 3×3 kernel → 3×3)")

In [ ]:
# Real image convolution with TensorFlow
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Create a simple test image (gradient)
img = np.zeros((8, 8), dtype=np.float32)
img[2:6, 2:6] = 1.0   # white square in center

# Reshape for TF: [batch, height, width, channels]
img_tf = img.reshape(1, 8, 8, 1)

# Edge detection filter: shape [kH, kW, in_channels, out_channels]
edge_filter = np.array([[-1,-1,-1],[-1,8,-1],[-1,-1,-1]],
                        dtype=np.float32).reshape(3,3,1,1)

# Apply convolution
output = tf.nn.conv2d(img_tf, edge_filter, strides=[1,1,1,1], padding='VALID')
output_np = output.numpy().squeeze()

fig, axes = plt.subplots(1, 2, figsize=(8,4))
axes[0].imshow(img, cmap='gray'); axes[0].set_title(f'Input {img.shape}')
axes[1].imshow(output_np, cmap='gray'); axes[1].set_title(f'After Conv {output_np.shape}')
plt.tight_layout(); plt.show()
print(f"Input: {img.shape} → Output: {output_np.shape}")
print("The filter detected the EDGES of the white square!")

## 🗣️ Interview Q&A

**Q: What is a convolution operation?**
> Sliding a small filter (kernel) over an input image, computing element-wise multiplication + sum at each position, producing a feature map that highlights specific patterns.

**Q: Why does CNN use convolution instead of fully connected layers?**
> Three reasons: (1) **Parameter sharing** — same filter scans whole image (much fewer params), (2) **Translation invariance** — detects a cat regardless of where it is, (3) **Local connectivity** — each neuron looks at small local region only.

**Q: What does each layer of a CNN learn?**
> Early layers → simple features (edges, colors), Middle layers → shapes, textures, Deep layers → complex objects, faces, parts of cars

**Q: Output size formula after convolution?**
> `Output = (Input - Kernel + 2×Padding) / Stride + 1`